In [ ]:
# --- Cell 0.1: Залежності ---
!pip install -q "sentence-transformers>=3.0" qdrant-client pandas numpy scikit-learn "transformers>=4.41.0" accelerate

# --- Cell 0.2: Імпорти ---
import os
import glob
import json
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder import (
    CrossEncoderTrainer,
    CrossEncoderTrainingArguments,
)
from sentence_transformers.cross_encoder.losses import (
    CachedMultipleNegativesRankingLoss,
    BinaryCrossEntropyLoss,
)
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator
from datasets import Dataset

from qdrant_client import QdrantClient
from qdrant_client.models import Filter

from google.colab import drive, userdata

drive.mount("/content/drive")

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"GPU: {device_name}")
USE_BF16 = any(x in device_name for x in ["L4", "A100", "H100", "L40", "A10"])
print(f"bf16: {USE_BF16}")

# --- Cell 0.3: Конфіг ---
CFG = {
    # Student модель
    "student_model": "BAAI/bge-reranker-v2-m3",

    # Дані
    "golden_dir": "/content/drive/MyDrive/CV_rank_Datasets/GoldenV2",
    "train_pairs": "/content/drive/MyDrive/CV_rank_Datasets/train_pairs.csv",
    "auto_eval_pool": "/content/drive/MyDrive/CV_rank_Datasets/auto_eval_pool.csv",

    # Qdrant колекції (нові, з trained embedder)
    "collection_jobs": "trained_embedder_jobs",
    "collection_cvs": "trained_embedder_cvs",

    # Тренування
    "output_dir": "/content/drive/MyDrive/bge-reranker-cv-finetuned",
    "epochs": 3,
    "batch_size": 16,       # L4 витягне; якщо OOM → 8
    "lr": 1e-5,
    "max_length": 512,

    # Дистиляція (опційно — вмикай якщо хочеш додатковий signal)
    "use_distillation": True,

    # Split
    "eval_fraction": 0.2,   # 20% golden вакансій у held-out eval
    "seed": 42,
}

# Qwen prompt-обгортка (така сама була при генерації train_pairs)
# ВАЖЛИВО: reranker бачить чистий текст, без Instruct-префіксу.
# Приберемо його при завантаженні train_pairs.
print("Config готовий")

# --- Cell 0.4: Qdrant підключення ---
client = QdrantClient(
    url=userdata.get("QDRANT_URL"),
    api_key=userdata.get("QDRANT_API_KEY"),
    timeout=300,
)
print("Колекції:", [c.name for c in client.get_collections().collections])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA L4
bf16: True
Config готовий
Колекції: ['candidates', 'jobs', 'trained_embedder_cvs', 'trained_embedder_jobs']


In [ ]:
import os
path = "/content/drive/MyDrive/CV_rank_Datasets/GoldenV2"
print("Існує:", os.path.exists(path))
if os.path.exists(path):
    files = os.listdir(path)
    print(f"Файлів: {len(files)}")
    for f in files[:5]:
        print(f"  {repr(f)}")  # repr щоб побачити спецсимволи

Існує: True
Файлів: 52
  'Copy of eval_pool_Software_Architect_Final.csv'
  'Copy of eval_pool_Tech_Lead___Backend_Final.csv'
  'Copy of eval_pool_Інженер_механік__CAD__Kinemati_Final.csv'
  'Copy of eval_pool_Software_Engineer_Final.csv'
  'Copy of eval_pool_Unity_Developer_Final.csv'


In [ ]:
# Перемонтуй Drive у Colab
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Перевір
import os
print(os.path.exists("/content/drive/MyDrive/CV_rank_Datasets/GoldenV2"))

Mounted at /content/drive
True


In [ ]:
# --- Cell 1.1: Завантаження train_pairs + чистка Instruct-префіксу ---
def strip_instruct(text):
    """
    train_pairs зроблені з Qwen-обгорткою:
      'Instruct: ...\nQuery: <справжній текст>'
    Reranker цього не треба — витягуємо чистий текст після 'Query:'.
    """
    text = str(text)
    if "Query:" in text:
        return text.split("Query:", 1)[1].strip()
    return text.strip()


train_pairs = pd.read_csv(CFG["train_pairs"])
train_pairs["vacancy_text"] = train_pairs["vacancy_text"].apply(strip_instruct)

# Обрізаємо занадто довгі тексти (reranker має max_length ліміт)
def clip(text, n=2000):
    return str(text)[:n]

train_pairs["vacancy_text"] = train_pairs["vacancy_text"].apply(clip)
train_pairs["cv_text_positive"] = train_pairs["cv_text_positive"].apply(clip)
train_pairs["cv_text_hard_negative"] = train_pairs["cv_text_hard_negative"].apply(clip)

print(f"Train pairs: {len(train_pairs)}")
print(f"\nПриклад vacancy_text:\n{train_pairs['vacancy_text'].iloc[0][:200]}")

# --- Cell 1.2: Формат для pairwise loss ---
# CachedMultipleNegativesRankingLoss очікує (query, positive, negative).
# Positive скориться вище за negative — це і є pairwise ranking.
train_dataset = Dataset.from_dict({
    "query": train_pairs["vacancy_text"].tolist(),
    "positive": train_pairs["cv_text_positive"].tolist(),
    "negative": train_pairs["cv_text_hard_negative"].tolist(),
})
print(f"Train dataset: {len(train_dataset)} тріплетів")



Train pairs: 1209

Приклад vacancy_text:
Title: Інженер з якості (Quality Engineer)
Company: Eleven
Category: 
Skills: 
Languages: 
Experience: 3 years experience
Employment: FULL_TIME
Location: 
Domain: Other

Description:
Eleven — рекрутин
Train dataset: 1209 тріплетів


In [ ]:
import subprocess
result = subprocess.run(
    ["find", "/content/drive/Shareddrives", "-name", "train_pairs*", "-o", "-name", "auto_eval*"],
    capture_output=True, text=True
)
print(result.stdout or "нема")

# Або якщо це не Shared drive, а просто розшарена папка:
result = subprocess.run(
    ["ls", "/content/drive/"],
    capture_output=True, text=True
)
print("Що є в /content/drive/:")
print(result.stdout)

нема
Що є в /content/drive/:
MyDrive
Shareddrives



In [ ]:
# --- Cell 2.1: Кеш вакансій з Qdrant по job_id ---
_vacancy_cache = {}

def get_vacancy_text(job_id):
    """Тягне текст вакансії з Qdrant по job_id (= point id)."""
    if job_id in _vacancy_cache:
        return _vacancy_cache[job_id]
    try:
        points = client.retrieve(
            collection_name=CFG["collection_jobs"],
            ids=[job_id],
            with_payload=True,
        )
        if points:
            payload = points[0].payload
            # embed_text містить повний опис; чистимо Instruct-префікс
            text = payload.get("embed_text", "")
            text = strip_instruct(text) if text else ""
            # Fallback якщо embed_text порожній
            if not text:
                text = f"{payload.get('title', '')} {payload.get('skills_tags', '')} {payload.get('experience_text', '')}"
            _vacancy_cache[job_id] = clip(text, 2000)
            return _vacancy_cache[job_id]
    except Exception as e:
        print(f"  ⚠ {job_id}: {e}")
    _vacancy_cache[job_id] = ""
    return ""


# --- Cell 2.2: Збірка golden з усіх CSV ---
golden_files = sorted(glob.glob(f"{CFG['golden_dir']}/*eval_pool_*.csv"))
print(f"Знайдено {len(golden_files)} golden файлів")

golden = {}  # job_id → {vacancy_text, candidates: [(cv_text, label)]}

for path in tqdm(golden_files, desc="Читання golden"):
    df = pd.read_csv(path)
    if "job_id" not in df.columns or "cv_text" not in df.columns:
        print(f"  ⚠ пропущено {os.path.basename(path)} — немає потрібних колонок")
        continue

    job_id = df["job_id"].iloc[0]
    vacancy_text = get_vacancy_text(job_id)
    if not vacancy_text:
        print(f"  ⚠ немає тексту вакансії для {os.path.basename(path)}")
        continue

    candidates = []
    for _, row in df.iterrows():
        cv = clip(row.get("cv_text", ""), 2000)
        label = int(row.get("Relevance_Score_0_to_3", 0))
        if cv:
            candidates.append((cv, label))

    if candidates:
        golden[job_id] = {
        "title": os.path.basename(path)
            .replace("Copy of ", "")
            .replace("eval_pool_", "")
            .replace("_Final.csv", ""),
        "vacancy_text": vacancy_text,
        "candidates": candidates,
    }

print(f"\nЗібрано {len(golden)} вакансій з валідними даними")
total_cands = sum(len(v["candidates"]) for v in golden.values())
print(f"Всього кандидатів: {total_cands}")


from collections import Counter
all_labels = []
for data in golden.values():
    all_labels.extend([c[1] for c in data["candidates"]])
print("Розподіл labels:", Counter(all_labels))

# --- Cell 2.3: Train/eval split по ВАКАНСІЯХ ---
# Критично: ділимо по вакансіях, не по парах, щоб eval-вакансії
# не текли в train. Golden йде ТІЛЬКИ в eval — це наша ground truth.
np.random.seed(CFG["seed"])
job_ids = list(golden.keys())
np.random.shuffle(job_ids)

n_eval = max(1, int(len(job_ids) * CFG["eval_fraction"]))
eval_job_ids = job_ids[:n_eval]
# Решта golden не використовується в train (train іде з train_pairs.csv),
# але можна тримати як додатковий eval / sanity check
holdout_job_ids = job_ids[n_eval:]

print(f"Eval вакансій: {len(eval_job_ids)}")
print(f"Holdout (не в train, не в eval): {len(holdout_job_ids)}")

golden_eval = {jid: golden[jid] for jid in eval_job_ids}

Знайдено 52 golden файлів


Читання golden:   0%|          | 0/52 [00:00<?, ?it/s]


Зібрано 52 вакансій з валідними даними
Всього кандидатів: 3256
Розподіл labels: Counter({0: 1558, 1: 1293, 2: 357, 3: 48})
Eval вакансій: 10
Holdout (не в train, не в eval): 42


In [ ]:
# --- Cell 3.1: NDCG helper ---
from sklearn.metrics import ndcg_score

def evaluate_reranker(model, golden_subset, k=10, batch_size=64):
    """
    Рахує NDCG@k і MRR для reranker на golden наборі.
    Для кожної вакансії: скорить усіх кандидатів, порівнює порядок з labels.
    """
    ndcgs, mrrs = [], []

    for job_id, data in golden_subset.items():
        vacancy = data["vacancy_text"]
        cvs = [c[0] for c in data["candidates"]]
        labels = np.array([c[1] for c in data["candidates"]])

        if labels.max() == 0:  # немає жодного релевантного — пропускаємо
            continue

        # Скоримо всі пари (vacancy, cv)
        pairs = [[vacancy, cv] for cv in cvs]
        scores = model.predict(pairs, batch_size=batch_size, show_progress_bar=False)

        # NDCG@k
        ndcg = ndcg_score([labels], [scores], k=k)
        ndcgs.append(ndcg)

        # MRR — позиція першого релевантного (label>=2)
        order = np.argsort(scores)[::-1]
        for rank, idx in enumerate(order, 1):
            if labels[idx] >= 2:
                mrrs.append(1.0 / rank)
                break
        else:
            mrrs.append(0.0)

    return {
        "ndcg@10": np.mean(ndcgs) if ndcgs else 0.0,
        "mrr": np.mean(mrrs) if mrrs else 0.0,
        "n_vacancies": len(ndcgs),
    }


# --- Cell 3.2: Baseline вимір ---
print("Завантаження baseline bge-reranker-v2-m3...")
baseline_model = CrossEncoder(CFG["student_model"], max_length=CFG["max_length"])

baseline_metrics = evaluate_reranker(baseline_model, golden_eval, k=10)
print(f"\n=== BASELINE (до тренування) ===")
print(f"NDCG@10: {baseline_metrics['ndcg@10']:.4f}")
print(f"MRR:     {baseline_metrics['mrr']:.4f}")
print(f"Вакансій в eval: {baseline_metrics['n_vacancies']}")

Завантаження baseline bge-reranker-v2-m3...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]


=== BASELINE (до тренування) ===
NDCG@10: 0.3837
MRR:     0.4928
Вакансій в eval: 10


In [ ]:
# --- Cell 4.1: Модель для тренування ---
model = CrossEncoder(
    CFG["student_model"],
    max_length=CFG["max_length"],
    num_labels=1,  # reranker видає один relevance score
)
print(f"Student: {CFG['student_model']}")

# --- Cell 4.2: Loss ---
# CachedMultipleNegativesRankingLoss — pairwise ranking з in-batch negatives.
# "Cached" версія дозволяє великі batch на обмеженій пам'яті.
loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=8)
print("Loss: CachedMultipleNegativesRankingLoss (pairwise)")

# --- Cell 4.3: Evaluator під час тренування ---
# CrossEncoderRerankingEvaluator міряє NDCG на golden під час навчання.
eval_samples = []
for job_id, data in golden_eval.items():
    positives = [c[0] for c in data["candidates"] if c[1] >= 2]
    negatives = [c[0] for c in data["candidates"] if c[1] == 0]
    if positives and negatives:
        eval_samples.append({
            "query": data["vacancy_text"],
            "positive": positives,
            "negative": negatives,
        })

reranking_evaluator = None
if eval_samples:
    reranking_evaluator = CrossEncoderRerankingEvaluator(
        samples=eval_samples,
        at_k=10,
        name="golden-eval",
        show_progress_bar=False,
    )
    print(f"Evaluator: {len(eval_samples)} вакансій з pos+neg")
else:
    print("⚠ Замало pos/neg для evaluator — тренуємось без online eval")

# --- Cell 4.4: Training arguments ---
args = CrossEncoderTrainingArguments(
    output_dir=CFG["output_dir"],
    num_train_epochs=CFG["epochs"],
    per_device_train_batch_size=CFG["batch_size"],
    learning_rate=CFG["lr"],
    warmup_ratio=0.1,

    # Точність під L4
    fp16=not USE_BF16,
    bf16=USE_BF16,

    # Стабільність
    max_grad_norm=1.0,
    dataloader_num_workers=2,

    # Eval / save
    eval_strategy="epoch" if reranking_evaluator else "no",
    save_strategy="epoch",
    logging_steps=20,
    save_total_limit=2,
    load_best_model_at_end=bool(reranking_evaluator),
    metric_for_best_model="golden-eval_ndcg@10" if reranking_evaluator else None,
    greater_is_better=True,
    seed=CFG["seed"],
)

# --- Cell 4.5: Тренування ---
trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
    evaluator=reranking_evaluator,
)

print("Запуск тренування...")
trainer.train()

model.save_pretrained(CFG["output_dir"])
print(f"Модель збережена → {CFG['output_dir']}")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Student: BAAI/bge-reranker-v2-m3
Loss: CachedMultipleNegativesRankingLoss (pairwise)
Evaluator: 10 вакансій з pos+neg
Запуск тренування...


Epoch,Training Loss,Validation Loss,Golden-eval Map,Golden-eval Mrr@10,Golden-eval Ndcg@10
1,0.695370,No log,0.509812,0.778333,0.549183
2,0.272485,No log,0.544274,0.816667,0.578455
3,0.350140,No log,0.557794,0.866667,0.591491


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель збережена → /content/drive/MyDrive/bge-reranker-cv-finetuned


In [ ]:
# Якщо CFG["use_distillation"] — донавчаємо на soft scores від Qwen.
# auto_eval_pool.csv має (vacancy, cv_text, auto_score 0-1).
# Reranker вчиться відтворювати ці scores через regression.

# --- Cell 4B.1: Distillation дані ---
if CFG["use_distillation"]:
    auto_df = pd.read_csv(CFG["auto_eval_pool"])
    auto_df["cv_text"] = auto_df["cv_text"].apply(lambda x: clip(x, 2000))
    # vacancy тут — це title; підтягнемо повний текст якщо треба,
    # але для дистиляції достатньо того що є в auto_score парі
    print(f"Distillation пар: {len(auto_df)}")
    print(f"auto_score: mean={auto_df['auto_score'].mean():.3f}, "
          f"min={auto_df['auto_score'].min():.3f}, max={auto_df['auto_score'].max():.3f}")

    # Формат для BinaryCrossEntropyLoss: (query, doc, label 0-1)
    distill_dataset = Dataset.from_dict({
        "query": auto_df["vacancy"].astype(str).tolist(),
        "doc": auto_df["cv_text"].tolist(),
        "label": auto_df["auto_score"].astype(float).tolist(),
    })
 # ПЕРЕД distill_args
import os, gc, torch

# Видалити старі об'єкти
if 'trainer' in dir():
    del trainer
if 'distill_trainer' in dir():
    del distill_trainer
if 'loss' in dir():
    del loss

# Перевести модель у eval і CPU тимчасово щоб очистити GPU
model.model.cpu()
gc.collect()
torch.cuda.empty_cache()
gc.collect()

print(f"Free VRAM: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")
# Має бути >18 GB free

# Повертаємо модель на GPU
model.model.to("cuda")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Cell 4B.2 — заміни distill_args
import os, gc, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()

distill_args = CrossEncoderTrainingArguments(
    output_dir=CFG["output_dir"] + "-distilled",
    num_train_epochs=2,
    per_device_train_batch_size=2,          # ще менше
    gradient_accumulation_steps=8,           # ефективний batch=16
    gradient_checkpointing=False,            # ← ВИМКНУТО
    learning_rate=5e-6,
    warmup_steps=50,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    max_grad_norm=1.0,
    dataloader_num_workers=0,
    logging_steps=20,
    eval_strategy="no",
    save_strategy="epoch",
    save_total_limit=2,
    seed=CFG["seed"],
    metric_for_best_model="loss"
)

distill_trainer = CrossEncoderTrainer(
    model=model,
    args=distill_args,
    train_dataset=distill_dataset,
    loss=distill_loss,
)

model.max_length = 384  # було 512
print("Дистиляція (з gradient checkpointing)...")
distill_trainer.train()
model.save_pretrained(CFG["output_dir"] + "-distilled")

Distillation пар: 1134
auto_score: mean=0.253, min=0.000, max=0.992
Free VRAM: 21.37 GB
Дистиляція (з gradient checkpointing)...


Step,Training Loss
20,0.509677
40,0.493271
60,0.498517
80,0.519575
100,0.488944
120,0.506873
140,0.493967


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# --- Cell 5.1: Fine-tuned vs baseline ---
finetuned_metrics = evaluate_reranker(model, golden_eval, k=10)

print("=" * 50)
print("РЕЗУЛЬТАТИ на held-out golden eval")
print("=" * 50)
print(f"{'Метрика':<12} {'Baseline':<12} {'Fine-tuned':<12} {'Δ':<10}")
print("-" * 50)
for metric in ["ndcg@10", "mrr"]:
    b = baseline_metrics[metric]
    f = finetuned_metrics[metric]
    print(f"{metric:<12} {b:<12.4f} {f:<12.4f} {f-b:+.4f}")
print(f"\nВакансій в eval: {finetuned_metrics['n_vacancies']}")

РЕЗУЛЬТАТИ на held-out golden eval
Метрика      Baseline     Fine-tuned   Δ         
--------------------------------------------------
ndcg@10      0.3837       0.4754       +0.0917
mrr          0.4928       0.5084       +0.0156

Вакансій в eval: 10


In [ ]:
# ============================================================================
#  SECTION 6 — ДЕПЛОЙ
# ============================================================================
#
#  Використання fine-tuned reranker у продакшн pipeline:
#
#  from sentence_transformers import CrossEncoder
#  reranker = CrossEncoder("/content/drive/MyDrive/bge-reranker-cv-finetuned")
#
#  # У pipeline: dense_search дає top-30, reranker перераховує
#  candidates = dense_search(vacancy, limit=30)  # з Qdrant
#  pairs = [[vacancy_text, c.payload["cv_text"]] for c in candidates]
#  scores = reranker.predict(pairs)
#  ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
#
#  Якщо distilled версія показала кращий NDCG — використовуй її:
#  reranker = CrossEncoder("/content/drive/MyDrive/bge-reranker-cv-finetuned-distilled")
# ============================================================================